# Alzheimer's MRI Classification — ResNet50 Transfer Learning

This notebook follows the structure of the working Base CNN notebook and uses pretrained ResNet50 with ImageNet weights.

In [ ]:
import os
import numpy as np
from pathlib import Path
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4


## 1. Locate dataset folders

In [ ]:
project_root = Path.cwd()
if not (project_root / 'Data' / 'Alzheimerr' / 'train_set').exists():
    project_root = project_root.parent

train_dir = project_root / 'Data' / 'Alzheimerr' / 'train_set'
test_dir = project_root / 'Data' / 'Alzheimerr' / 'test_set'
output_dir = project_root / 'Outputs'

if not train_dir.exists() or not test_dir.exists():
    raise FileNotFoundError(
        f"Dataset folders not found. Checked: {train_dir} and {test_dir}"
    )

output_dir.mkdir(parents=True, exist_ok=True)
print("Train directory:", train_dir)
print("Test directory:", test_dir)
print("Output directory:", output_dir)


## 2. Create ResNet50 data generators

ResNet50 requires its own `preprocess_input` function. Do not use `rescale=1./255` with this preprocessing.

In [ ]:
train_datagen_resnet = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.1,
    validation_split=0.2
)

validation_datagen_resnet = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

test_datagen_resnet = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_gen_resnet = train_datagen_resnet.flow_from_directory(
    str(train_dir),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=SEED,
    shuffle=True
)

val_gen_resnet = validation_datagen_resnet.flow_from_directory(
    str(train_dir),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=SEED,
    shuffle=False
)

test_gen_resnet = test_datagen_resnet.flow_from_directory(
    str(test_dir),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print("Class indices:", train_gen_resnet.class_indices)
print(f"Train samples: {train_gen_resnet.samples}")
print(f"Validation samples: {val_gen_resnet.samples}")
print(f"Test samples: {test_gen_resnet.samples}")

if train_gen_resnet.num_classes != NUM_CLASSES:
    raise ValueError(
        f"Expected {NUM_CLASSES} classes, found {train_gen_resnet.num_classes}. "
        "Check the class subfolders in train_set."
    )


## 3. Build and compile ResNet50 transfer-learning model

In [ ]:
def build_resnet50_model(
    input_shape=(224, 224, 3),
    num_classes=4,
    learning_rate=1e-3
):
    resnet_base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )

    # Freeze pretrained convolutional layers for feature extraction
    resnet_base_model.trainable = False

    model = models.Sequential([
        layers.Input(shape=input_shape),
        resnet_base_model,
        layers.GlobalAveragePooling2D(),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


resnet50_model = build_resnet50_model()
resnet50_model.summary()


## 4. Train ResNet50 with EarlyStopping

In [ ]:
def train_transfer_model(
    model,
    train_gen,
    val_gen,
    epochs=40,
    patience=6
):
    callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=patience,
            restore_best_weights=True,
            verbose=1
        )
    ]

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=epochs,
        callbacks=callbacks,
        verbose=2  # One completed-epoch summary; avoids partial-batch progress output
    )
    return history


print("Training ResNet50...")
resnet50_history = train_transfer_model(
    resnet50_model,
    train_gen_resnet,
    val_gen_resnet,
    epochs=40,
    patience=6
)


## 5. Evaluate model on test set

In [ ]:
test_loss, test_accuracy = resnet50_model.evaluate(
    test_gen_resnet,
    verbose=1
)

print(f"ResNet50 Test Loss: {test_loss:.4f}")
print(f"ResNet50 Test Accuracy: {test_accuracy:.4f}")
print(f"ResNet50 Test Accuracy: {test_accuracy * 100:.2f}%")


## 6. Save model and training plot

In [ ]:
model_path = output_dir / 'resnet50_model.keras'
resnet50_model.save(model_path)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(resnet50_history.history['accuracy'], label='Training Accuracy')
plt.plot(resnet50_history.history['val_accuracy'], label='Validation Accuracy')
plt.title('ResNet50 Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(resnet50_history.history['loss'], label='Training Loss')
plt.plot(resnet50_history.history['val_loss'], label='Validation Loss')
plt.title('ResNet50 Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plot_path = output_dir / 'resnet50_training_plot.png'
plt.tight_layout()
plt.savefig(plot_path, dpi=150)
plt.show()

print(f"Model saved to: {model_path}")
print(f"Training plot saved to: {plot_path}")


## 7. Predict a single test image

In [ ]:
def predict_single_image_resnet(
    model,
    image_path,
    class_names=None,
    target_size=(224, 224)
):
    if class_names is None:
        # Use generator class order to ensure labels match model output
        class_names = [
            name for name, index in sorted(
                train_gen_resnet.class_indices.items(),
                key=lambda item: item[1]
            )
        ]

    img = tf.keras.preprocessing.image.load_img(
        image_path,
        target_size=target_size
    )
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = preprocess_input(img_array)
    img_array = np.expand_dims(img_array, axis=0)

    probabilities = model.predict(img_array, verbose=0)[0]
    predicted_index = int(np.argmax(probabilities))
    return class_names[predicted_index], probabilities


# Select the first image found in the test set
class_folder = next(
    folder for folder in test_dir.iterdir()
    if folder.is_dir() and any(folder.iterdir())
)
image_path = next(
    file for file in class_folder.iterdir()
    if file.is_file() and file.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}
)

predicted_class, probabilities = predict_single_image_resnet(
    resnet50_model,
    str(image_path)
)

print("Test image:", image_path)
print("Predicted class:", predicted_class)
print("Prediction probabilities:", probabilities)


## Notes

- ResNet50 input size is **224 × 224 × 3**.
- Use `preprocess_input` for training, validation, test, and single-image prediction.
- The final Dense layer has 4 outputs for the four MRI classes.
- The pretrained ResNet50 backbone is frozen in this feature-extraction stage.
- If you later fine-tune the backbone, unfreeze selected top layers and recompile with a much smaller learning rate.